In [25]:
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score
from sklearn.metrics import average_precision_score
from lightgbm import LGBMClassifier

In [26]:
# TODO:
# 1. use npz instead of csv (done)
# 2. check how much the metrics increase for each time you increase the amount of data (kinda done)
# 3. check which features are actually useful (in process)
# 4. neural network

In [27]:
data = np.load("data/moves.npz")

X, y, groups = data["X"], data["y"], data["groups"]

In [28]:
# Check which features are useful:

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)

train_idx, test_idx = next(gss.split(X, groups=groups))

for i in range(3):
    if i == 0:
        # clock, ratings, and side to move only features
        temp = X[:, 768:].copy()
    elif i == 1:
        # board only features
        temp = X[:, :768].copy()
    else:
        # every feature
        temp = X.copy()

    X_train = temp[train_idx]
    X_test = temp[test_idx]

    y_train = y[train_idx]
    y_test = y[test_idx]

    model = LGBMClassifier(verbosity=-1)

    model.fit(X_train, y_train)

    y_prob = model.predict_proba(X_test)[:, 1]
    auc_score = roc_auc_score(y_test, y_prob)
    ap_score = average_precision_score(y_test, y_prob)

    if i == 0:
        print("Clock, ratings, side to move, material difference, number of legal moves, and hanging pieces only features")
        print(f"Roc Auc: {auc_score:0.4f}")
        print(f"Average Precision: {ap_score:0.4f}\n")

    elif i == 1:
        print("Board only features")
        print(f"Roc Auc: {auc_score:0.4f}")
        print(f"Average Precision: {ap_score:0.4f}\n")

    else:
        print("Every feature")
        print(f"Roc Auc: {auc_score:0.4f}")
        print(f"Average Precision: {ap_score:0.4f}\n")

Clock, ratings, side to move, material difference, number of legal moves, and hanging pieces only features
Roc Auc: 0.6887
Average Precision: 0.1482

Board only features
Roc Auc: 0.6230
Average Precision: 0.1197

Every feature
Roc Auc: 0.7123
Average Precision: 0.1627

